In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start), COUNT(*)
FROM prod.detection.viewing_content_firehose
WHERE session_start >= CURRENT_DATE
GROUP BY 1

In [0]:
schema = 'dev.mohit_gangwani'
report_name = 'cure_viewing_content_golden_table'
start_time = '2025-07-14 14:00:00'
end_time = '2025-07-14 15:00:00'

In [0]:
spark.sql(f"""DROP TABLE IF EXISTS {schema}.{report_name};""")

In [0]:
%sql
SELECT app_name, 'acr blocked' FROM prod.detection.app_activity_distribution_blacklist
UNION
SELECT app_name, 'obfuscated' FROM prod.detection.app_viewing_distribution_blacklist

In [0]:
spark.sql(f"""
CREATE TABLE {schema}.{report_name} AS
WITH activity_obfuscation AS (
  SELECT blocked_apps.app_name, override.client_name
  FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
  LEFT JOIN prod.detection.app_customer_activity_distribution_override override
    ON blocked_apps.app_name = override.app_name
  GROUP BY 1, 2
),
viewing_obfuscation AS (
  SELECT blocked_apps.app_name, override.client_name
  FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
  LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
    ON blocked_apps.app_name = override.app_name
  GROUP BY 1, 2
)
, nielsen_replacement_national_nyc_alias AS (
  SELECT rl.station_id, rl.fk_show_id, rl.tuner_channel_id, rl.tuner_program_id, rl.airdate
  FROM prod.detection.nielsen_replacement_national_nyc AS rl
  JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
    ON bl.station_id = rl.station_id
   AND bl.blacklist_end >= '{start_time}'
  GROUP BY ALL
)
, nielsen_replacement_local_alias AS (
  SELECT rl.station_id, rl.fk_show_id, rl.dma_id, rl.tuner_channel_id, rl.tuner_program_id, rl.airdate
  FROM prod.detection.nielsen_replacement_local AS rl
  JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
    ON bl.station_id = rl.station_id
   AND bl.blacklist_end >= '{start_time}'
  GROUP BY ALL
)
, vod_stations AS (
  SELECT station_id, vendor_name
  FROM prod.detection.station_distribution_obfuscation_overwrite
  GROUP BY 1, 2
)
, inscape_station_map_dedupe AS (
  SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, channel_affiliate
  FROM (
    SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
    , CASE WHEN st.inscape_station_name IS NOT NULL THEN st.inscape_station_name
           WHEN LOWER(st.station_affil) LIKE '%affiliate%'
             OR LOWER(st.station_affil) LIKE '%independent%'
             OR LOWER(st.station_affil) LIKE '%low power%' THEN st.station_affil
      END AS channel_affiliate
    , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
     AND st.vendor_name = ism.mapped_vendor
  ) ism
  WHERE ism.rn = 1
)
SELECT tvid, fk_tvid, zipcode, dma
, tms_episode_id, tivo_episode_id, tms_title, tivo_title, tms_airdate, tivo_airdate
, tms_channel_callsign, tivo_channel_callsign
, mt_start, session_start, session_end
, tms_channel_affiliate, tivo_channel_affiliate, is_live
, ip_address, input_category, input_device, app_service
, tuner_tms_episode_id, tuner_tivo_episode_id, tuner_tms_title, tuner_tivo_title, tuner_tms_airdate, tuner_tivo_airdate
, tuner_tms_channel_callsign, tuner_tivo_channel_callsign
, tuner_mt_start, tuner_tms_channel_affiliate, tuner_tivo_channel_affiliate, tuner_is_live
, tuner_input_category, tuner_input_device, tuner_app_service, tuner_channel_number
, enableaudioacr, dma_code
, vizio_epg_channel_id, vizio_epg_program_id, tms_show_genre, tivo_show_genre
, tms_epi_title, tivo_epi_title, series_id, show_duration
, vizio_epg_not_null, nielsen_exclusive, content_only_condition, tuner_content_only_condition, vod_station
, '|'||array_join(collect_set(acrb_client), '|')||'|' AS acrb_clients
, '|'||array_join(collect_set(appb_client), '|')||'|' AS appb_clients
, '|'||array_join(collect_set(client_id), '|')||'|' AS client_id_not_null
FROM (
  SELECT DISTINCT COALESCE(tv.long_tvid, tv.vizio_tvid) AS tvid
  , c.fk_tvid
  , NULLIF(location.zipcode, '') AS zipcode
  , REPLACE(dma.dma_name, ',', '') AS dma
  ------------------ Episode ID -------------------
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN
          CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
               ELSE vizio_program.program_tms_id END
         WHEN c.file_ingested = true THEN COALESCE(md.external_id,SPLIT(cid.content_cid, '_')[0])
         ELSE tms_show.database_key
    END AS tms_episode_id
  
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN NULL
         WHEN c.file_ingested = true THEN COALESCE(md.external_id,SPLIT(cid.content_cid, '_')[0])
         ELSE tivo_show.database_key
    END AS tivo_episode_id
  ------------------------------------------------------------
  ------------------ Show Title -------------------
  , REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN
          CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
               WHEN vizio_program.series_aggregate_title IS NOT NULL AND vizio_program.series_aggregate_title != '' THEN vizio_program.series_aggregate_title
               ELSE vizio_program.title END
         WHEN c.file_ingested THEN NULL
         ELSE tms_show.title
    END, '[\",]', '') AS tms_title
  
  , REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN
          CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
               WHEN vizio_program.series_aggregate_title IS NOT NULL AND vizio_program.series_aggregate_title != '' THEN vizio_program.series_aggregate_title
               ELSE vizio_program.title END
         WHEN c.file_ingested THEN NULL
         ELSE tivo_show.title, '[\",]', '')
    END AS tivo_title
  ------------------------------------------------------------
  ------------------ Airdate -------------------
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN c.vizio_epg_station IS NOT NULL THEN c.tms_airdate
         WHEN cid.content_cid = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
         ELSE c.tms_airdate
    END AS tms_airdate
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
         WHEN c.vizio_epg_station IS NOT NULL THEN c.airdate
         WHEN cid.content_cid = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
         ELSE c.airdate
    END AS tivo_airdate
  ------------------------------------------------------------
  ------------------ Channel Call Sign -------------------
  , CASE WHEN tms_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.vizio_epg_station IS NOT NULL THEN tms_map.inscape_call_sign
         WHEN c.file_ingested = true THEN SPLIT(cid.content_cid, '_')[1]
         ELSE tms_map.inscape_call_sign
    END AS tms_channel_callsign
  , CASE WHEN tivo_station_obfs.station_id IS NOT NULL THEN NULL
         WHEN c.vizio_epg_station IS NOT NULL THEN tivo_map.inscape_call_sign
         WHEN c.file_ingested = true THEN SPLIT(cid.content_cid, '_')[1]
         ELSE tivo_map.inscape_call_sign
    END AS tivo_channel_callsign
  ------------------------------------------------------------
  ------------------ Media Time Start -------------------
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL 
         WHEN c.vizio_epg_station IS NOT NULL THEN c.media_time_start
         WHEN cid.content_cid = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
         ELSE LEAST(c.media_time_start, c.runtime)
    END AS mt_start
  ------------------------------------------------------------
  , c.session_start
  , c.session_end
  ------------------------------------------------------------
  ------------------ Station Affiliate -------------------
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN
             CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED' 
                  ELSE vizio_station.name END
         WHEN tms_station_obfs.station_id IS NOT NULL THEN NULL
         ELSE tms_map.channel_affiliate
    END AS tms_channel_affiliate
  
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN
             CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN 'OBFUSCATED' 
                  ELSE vizio_station.name END
         WHEN tivo_station_obfs.station_id IS NOT NULL THEN NULL
         ELSE tivo_map.channel_affiliate
    END AS tivo_channel_affiliate
  ------------------------------------------------------------
  ------------------ Live -------------------
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN 't'
         WHEN cid.content_cid = 'unknown' AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN NULL
         WHEN c.is_live = TRUE THEN 't'
         WHEN c.is_live = FALSE THEN 'f'
    END AS is_live
  ------------------------------------------------------------
  , ip.ip_address
  , tvis.category AS input_category
  , tvis.input_device
  , CASE WHEN UPPER(tvis.category) = 'APPS' THEN
          CASE WHEN c.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
               WHEN c.vizio_epg_station IS NULL and tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
               WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora',  'tv games') AND cid.content_cid <> 'unknown' THEN NULL
               WHEN lower(tis.app_name) = 'unknown' THEN NULL
               ELSE tis.app_name
          END
         WHEN c.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4',  'playstation 5','roku') THEN 'vMVPD'
    END AS app_service
  ---------------- Tuner Columns ----------------
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_show.database_key END  AS tuner_tms_episode_id
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_show.database_key END AS tuner_tivo_episode_id
  , CASE WHEN c.file_ingested THEN NULL ELSE REGEXP_REPLACE(tuner_tms_show.title, '[\",]', '')  END AS tuner_tms_title
  , CASE WHEN c.file_ingested THEN NULL ELSE REGEXP_REPLACE(tuner_tivo_show.title, '[\",]', '') END AS tuner_tivo_title
  , CASE WHEN c.file_ingested THEN NULL ELSE c.tms_airdate END AS tuner_tms_airdate
  , CASE WHEN c.file_ingested THEN NULL ELSE c.airdate     END AS tuner_tivo_airdate
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tms_map.inscape_call_sign  END AS tuner_tms_channel_callsign
  , CASE WHEN c.file_ingested THEN NULL ELSE tuner_tivo_map.inscape_call_sign END AS tuner_tivo_channel_callsign
  , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
             THEN LEAST((unix_timestamp(c.session_start)-unix_timestamp(COALESCE(c.airdate, c.tms_airdate))), c.runtime) 
         ELSE LEAST(c.media_time_start, c.runtime)
    END AS tuner_mt_start
  , tuner_tms_map.channel_affiliate  AS tuner_tms_channel_affiliate
  , tuner_tivo_map.channel_affiliate AS tuner_tivo_channel_affiliate
  , CASE WHEN COALESCE(c.tuner_channel_id, NULLIF(c.tms_tuner_channel_id,98989898)) IS NOT NULL THEN 't'
         WHEN c.is_live = TRUE THEN 't'
         WHEN c.is_live = FALSE THEN 'f'
    END AS tuner_is_live
  , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL AND UPPER(tvis.category) in ('APPS', 'OTHER', 'OTT') THEN 'HD TV'
         WHEN UPPER(tvis.category) = 'OTHER' AND tis.app_name = 'WatchFree+' AND cid.content_cid != 'unknown' THEN 'HD TV' 
         WHEN UPPER(tvis.category) = 'OTT' AND tis.app_name = 'WatchFree+' THEN 'APPS'
         ELSE tvis.category
    END AS tuner_input_category
  , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL THEN 'OTA'
         WHEN inps.input_source = 'DTV' THEN 'OTA'
         ELSE tvis.input_device
    END AS tuner_input_device
  , CASE WHEN COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NOT NULL
          AND NVL(tvis.input_device, 'OTA') = 'OTA' THEN 'WatchFree+'
         WHEN UPPER(tvis.category) = 'APPS' THEN
            CASE WHEN c.vizio_epg_station IS NOT NULL THEN 'WatchFree+'
                 WHEN c.vizio_epg_station IS NULL AND tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
                 WHEN LOWER(tis.app_name) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND cid.content_cid <> 'unknown' THEN NULL
                 WHEN lower(tis.app_name) = 'unknown' THEN NULL
                 ELSE tis.app_name
            END
         WHEN c.is_live = true
          AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku')
          AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NULL THEN 'vMVPD'
         WHEN inps.input_source IN ('DTV', 'TUNER', 'COAXIAL', 'ATV')
          AND NVL(tvis.input_device, 'OTA') = 'OTA'
          AND tis.app_name ='WatchFree+'
          AND c.is_live = TRUE THEN 'WatchFree+'
    END AS tuner_app_service
  , c.tuner_channel_number
  ---------------Additional Fields ---------------------------
  , CASE WHEN settings.enableaudioacr = 1 THEN 't' ELSE 'f' END AS enableaudioacr
  , dma.dma_code AS dma_code
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
         ELSE vizio_station.station_id
    END AS vizio_epg_channel_id
  , CASE WHEN chanb.channel_name IS NOT NULL OR c.vizio_epg_station in ('98989898989898', '9898989898', '-1') THEN NULL
         ELSE vizio_program.program_aggregate_id
    END AS vizio_epg_program_id
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN REGEXP_REPLACE(vizio_program.aggregate_genres, '[\\[\\]]', '')
                ELSE tms_show.genre, ', ?', '|')
           END, ''
    ) AS tms_show_genre
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN REGEXP_REPLACE(vizio_program.aggregate_genres, '[\\[\\]]', '')
                ELSE tivo_show.genre, ', ?', '|')
           END, ''
    ) AS tivo_show_genre
  , NULLIF(CASE WHEN c.vizio_epg_station IS NOT NULL THEN vizio_program.title
                ELSE tms_show.epi_title, '[\",]', '')
           END, ''
    ) AS tms_epi_title
  , NULLIF(REGEXP_REPLACE(CASE WHEN c.vizio_epg_station IS NOT NULL THEN vizio_program.title
                ELSE tivo_show.epi_title, '[\",]', '')
           END, ''
    ) AS tivo_epi_title
  , CASE WHEN c.file_ingested THEN NULL ELSE tivo_show.series_id END AS series_id
  , CASE WHEN c.file_ingested THEN NULL ELSE c.runtime END AS show_duration
  ------------------ Conditions -------------------
  ------Bools-------
  , CASE WHEN c.vizio_epg_station IS NOT NULL THEN TRUE ELSE FALSE END AS vizio_epg_not_null
  , CASE WHEN COALESCE(tivo_nielsen_blacklist.station_id, tms_nielsen_blacklist.station_id) IS NOT NULL
          AND (COALESCE(tivo_rep_local.station_id, tivo_rep_nyc_nat.station_id, tms_rep_local.station_id, tms_rep_nyc_nat.station_id) IS NULL
               OR COALESCE(tms_nielsen_blacklist.ingest_time, tivo_nielsen_blacklist.ingest_time) IS NOT NULL) THEN TRUE
         ELSE FALSE
    END AS nielsen_exclusive
  , CASE WHEN cid.content_cid = 'unknown' AND vizio_station.name IS NULL THEN TRUE ELSE FALSE END AS content_only_condition
  , CASE WHEN cid.content_cid = 'unknown'
          AND vizio_station.name IS NULL
          AND COALESCE(c.tuner_channel_id, c.tms_tuner_channel_id) IS NULL THEN TRUE
         ELSE FALSE
    END AS tuner_content_only_condition
  , CASE WHEN COALESCE(tivo_vod_stations.station_id, tms_vod_stations.station_id) IS NOT NULL THEN TRUE ELSE FALSE END AS vod_station
  ------Later Aggs-------
  , CASE WHEN acrb.app_name IS NOT NULL AND c.vizio_epg_station IS NULL THEN CASE WHEN acrb.client_name IS NULL THEN 'ALL' ELSE acrb.client_name END
    END AS acrb_client
  , CASE WHEN appb.app_name IS NOT NULL AND c.vizio_epg_station IS NULL THEN CASE WHEN appb.client_name IS NULL THEN 'ALL' ELSE appb.client_name END
    END AS appb_client
  , cl.client_name AS client_id
  ------------------------------------------------------------
  -- Joins that do not need to be modified
  FROM prod.detection.viewing_content_firehose AS c
  JOIN prod.detection.zoo AS z
    ON c.fk_zoo_id = z.zoo_id
   AND z.zoo = 'control-zoo-dtsprod.tvinteractive.tv'
  JOIN prod.detection.tv AS tv
    ON c.fk_tvid = tv.tvid
   AND tv.oem = 'VIZIO'
  -- Location
  JOIN prod.detection.tv_settings AS tv_settings
    ON c.session_start >= tv_settings.create_timestamp
   AND c.session_start < tv_settings.next_create_timestamp
   AND tv_settings.create_timestamp <= '{end_time}'::timestamp
   AND tv_settings.next_create_timestamp >= '{start_time}'::timestamp
   AND c.fk_tvid = tv_settings.fk_tvid
  JOIN prod.detection.settings AS settings
    ON tv_settings.fk_settings_id = settings.settings_id
   AND UPPER(settings.country_name) = 'USA'
  JOIN prod.detection.tv_populations AS u
    ON c.fk_tvid = u.fk_tvid
  JOIN prod.detection.populations AS pop
    ON u.fk_population_id = pop.population_id
   AND pop.population_name = 'opted_in'
  JOIN prod.detection.location AS location
    ON c.fk_location_id = location.location_id
   AND UPPER(location.country_code) = 'US'
  LEFT OUTER JOIN prod.detection.dma AS dma
    ON c.fk_dma_id = dma.dma_id
  -- Content
  JOIN prod.detection.content_ids_firehose AS cid
    ON cid.content_id = c.fk_content_id
  -- IP Address
  LEFT OUTER JOIN prod.detection.tv_ip_address AS ip
    ON c.session_start >= ip.create_timestamp
   AND c.session_start < ip.next_create_timestamp
   AND ip.create_timestamp <= '{end_time}'::timestamp
   AND ip.next_create_timestamp >= '{start_time}'::timestamp
   AND tv.tvid = ip.fk_tvid
  -- Vizio Joins
  LEFT OUTER JOIN prod.detection.vizio_epg_station AS vizio_station
    ON TRY_CAST(c.vizio_epg_station AS STRING) = TRY_CAST(vizio_station.station_id AS STRING)
  LEFT OUTER JOIN prod.detection.vizio_epg_program_aggregate AS vizio_program
    ON TRY_CAST(c.vizio_epg_program AS STRING) = TRY_CAST(vizio_program.program_aggregate_id AS STRING)
   AND TRY_CAST(c.vizio_epg_program AS STRING) NOT IN ('0', '', '-1')
  LEFT OUTER JOIN prod.detection.free_channels_distribution_blacklist AS chanb
    ON vizio_station.name = chanb.channel_name
  -- Input Joins
  LEFT OUTER JOIN prod.detection.input_source AS inps
    ON c.fk_input_source_id = inps.input_source_id
  JOIN prod.detection.tv_input_stats_firehose AS tvis
    ON c.session_start >= tvis.create_timestamp
   AND c.session_start < tvis.next_create_timestamp
   AND tvis.create_timestamp <= '{end_time}'::timestamp
   AND tvis.next_create_timestamp >= '{start_time}'::timestamp
   AND  c.fk_tvid = tvis.fk_tvid
   AND  c.fk_input_source_id = tvis.fk_input_source_id
  LEFT OUTER JOIN prod.detection.tv_inputsource AS tis
    ON c.session_start >= (tis.create_timestamp::double)::timestamp
   AND c.session_start < (tis.next_create_timestamp::double)::timestamp
   AND c.fk_tvid = tis.fk_tvid
   AND c.fk_input_source_id = tis.fk_input_source_id
   AND tis.create_timestamp <= ('{end_time}'::timestamp::double)::timestamp
   AND tis.next_create_timestamp >= ('{start_time}'::timestamp::double)::timestamp
  -- App blacklist
  LEFT OUTER JOIN activity_obfuscation AS appb
    ON tis.app_name = appb.app_name
  LEFT OUTER JOIN viewing_obfuscation AS acrb
    ON tis.app_name = acrb.app_name
  ---------------------------------------------
  -- Client Specific
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS m
    ON m.fk_content_id = c.fk_content_id
  LEFT OUTER JOIN prod.detection.clients AS cl
    ON m.fk_client_id = cl.client_id
  LEFT OUTER JOIN prod.detection.content_id_external_firehose AS md
    ON md.fk_content_id = c.fk_content_id
  LEFT OUTER JOIN prod.detection.clients AS cli
    ON md.fk_client_id = cli.client_id
  ---------------------------------------------
  -- TiVo TMS specific joins
  LEFT OUTER JOIN inscape_station_map_dedupe AS tivo_map
    ON tivo_map.mapped_vendor_station_id = c.fk_station_id
   AND tivo_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tivo_show
    ON tivo_show.show_id = c.fk_show_id
   AND tivo_show.vendor_name = 'TIVO'
  
  LEFT OUTER JOIN prod.detection.epg_show AS tms_show
    ON tms_show.show_id = c.tms_show_id
   AND tms_show.vendor_name = 'TMS'
  LEFT OUTER JOIN inscape_station_map_dedupe AS tms_map
    ON tms_map.mapped_vendor_station_id = c.tms_station_id
   AND tms_map.mapped_vendor = 'TMS'
  ---------------------------------------------
  -- Tuner joins
  LEFT OUTER JOIN inscape_station_map_dedupe AS tuner_tivo_map
    ON tuner_tivo_map.mapped_vendor_station_id = c.tuner_channel_id
   AND tuner_tivo_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN prod.detection.epg_show AS tuner_tivo_show
    ON tuner_tivo_show.show_id = c.tuner_program_id
   AND tuner_tivo_show.vendor_name = 'TIVO'
  
  LEFT OUTER JOIN prod.detection.epg_show AS tuner_tms_show
    ON tuner_tms_show.show_id = c.tms_tuner_program_id
   AND tuner_tms_show.vendor_name = 'TMS'
  LEFT OUTER JOIN inscape_station_map_dedupe AS tuner_tms_map
    ON tuner_tms_map.mapped_vendor_station_id = c.tms_tuner_channel_id
   AND tuner_tms_map.mapped_vendor = 'TMS'
  ---------------------------------------------
  -- Blacklist Joins
  LEFT OUTER JOIN vod_stations AS tivo_vod_stations
    ON tivo_vod_stations.station_id = tivo_map.inscape_station_id
   AND tivo_vod_stations.vendor_name = 'TIVO'
  LEFT OUTER JOIN vod_stations AS tms_vod_stations
    ON tms_vod_stations.station_id = tms_map.inscape_station_id
   AND tms_vod_stations.vendor_name = 'TMS'

  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tivo_station_obfs
    ON tivo_station_obfs.vendor_station_id = tivo_map.inscape_station_id
  LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS tms_station_obfs
    ON tms_station_obfs.vendor_station_id = tms_map.inscape_station_id

  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tivo_nielsen_blacklist
    ON tivo_nielsen_blacklist.station_id = tivo_map.inscape_station_id
   AND c.session_start >= tivo_nielsen_blacklist.blacklist_start
   AND c.session_start < tivo_nielsen_blacklist.blacklist_end
  LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS tms_nielsen_blacklist
    ON tms_nielsen_blacklist.station_id = tms_map.inscape_station_id
   AND c.session_start >= tms_nielsen_blacklist.blacklist_start
   AND c.session_start < tms_nielsen_blacklist.blacklist_end

  LEFT OUTER JOIN nielsen_replacement_local_alias AS tivo_rep_local
    ON tivo_rep_local.station_id = tivo_map.inscape_station_id
   AND tivo_rep_local.airdate = c.airdate
   AND tivo_rep_local.fk_show_id = c.fk_show_id
   AND tivo_rep_local.dma_id = c.fk_dma_id
  LEFT OUTER JOIN nielsen_replacement_local_alias AS tms_rep_local
    ON tms_rep_local.station_id = tms_map.inscape_station_id
   AND tms_rep_local.airdate = c.tms_airdate
   AND tms_rep_local.fk_show_id = c.tms_show_id
   AND tms_rep_local.dma_id = c.fk_dma_id

  LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS tivo_rep_nyc_nat
    ON tivo_rep_nyc_nat.station_id = tivo_map.inscape_station_id
   AND tivo_rep_nyc_nat.airdate = c.airdate
   AND tivo_rep_nyc_nat.fk_show_id = c.fk_show_id
  LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS tms_rep_nyc_nat
    ON tms_rep_nyc_nat.station_id = tms_map.inscape_station_id
   AND tms_rep_nyc_nat.airdate = c.tms_airdate
   AND tms_rep_nyc_nat.fk_show_id =c.tms_show_id
  ---------------------------------------------
  WHERE c.session_start >= '{start_time}'::timestamp
    AND c.session_start < '{end_time}'::timestamp
    AND CASE c.file_ingested
      WHEN true THEN
          CASE NULLIF(SPLIT(cid.content_cid, '_')[1], '') IS NOT NULL AND NULLIF(SPLIT_PART(cid.content_cid, '_', 3), '') IS NULL
          WHEN true THEN SPLIT(cid.content_cid, '_')[1]
          ELSE NULL
          END
      ELSE COALESCE(tivo_map.inscape_call_sign, tms_map.inscape_call_sign, 'KeepSessionForNullReport')
      END NOT IN (SELECT DISTINCT chan_callsign FROM customer_reports.bad_chan_callsign)
)
GROUP BY tvid, fk_tvid, zipcode, dma
, tms_episode_id, tivo_episode_id, tms_title, tivo_title, tms_airdate, tivo_airdate
, tms_channel_callsign, tivo_channel_callsign
, mt_start, session_start, session_end
, tms_channel_affiliate, tivo_channel_affiliate, is_live
, ip_address, input_category, input_device, app_service
, tuner_tms_episode_id, tuner_tivo_episode_id, tuner_tms_title, tuner_tivo_title, tuner_tms_airdate, tuner_tivo_airdate
, tuner_tms_channel_callsign, tuner_tivo_channel_callsign
, tuner_mt_start, tuner_tms_channel_affiliate, tuner_tivo_channel_affiliate, tuner_is_live
, tuner_input_category, tuner_input_device, tuner_app_service, tuner_channel_number
, enableaudioacr, dma_code
, vizio_epg_channel_id, vizio_epg_program_id, tms_show_genre, tivo_show_genre
, tms_epi_title, tivo_epi_title, series_id, show_duration
, vizio_epg_not_null, nielsen_exclusive, content_only_condition, tuner_content_only_condition, vod_station;
""")

In [0]:
%sql
SELECT COUNT(*)
FROM dev.mohit_gangwani.cure_viewing_content_golden_table

In [0]:
%sql
SELECT *
FROM dev.mohit_gangwani.cure_viewing_content_golden_table
WHERE tuner_channel_number IS NOT NULL
LIMIT 100